In [1]:
import json
problems = []
solutions = []
outputs = []
gts = []
file_path = "/media/volume/EmbodiedAICompetition/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/gsm8k_train/10/0.1/10000/predictions.jsonl"
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        problems.append(item["problem"])
        solutions.append(item["solution"])
        outputs.append(item["model_generation"])
        
# Create prompt texts from problems



In [2]:
metric_path = "/media/volume/EmbodiedAICompetition/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/gsm8k_train/10/0.1/10000/metrics.json"
with open(metric_path, 'r') as f:
    json_output = json.load(f)

import numpy as np
true_n_ls = np.array(json_output['true_n_ls'])
print((true_n_ls))
# extract the percentage of correct answers for each question, total len(true_n_ls) trails, each trail has len(true_n_ls[0]) questions
correct_percentage = np.sum(true_n_ls, axis=0) / true_n_ls.shape[0]
print(correct_percentage)
only_one_correct = []
for i in range(len(correct_percentage)):
    if correct_percentage[i] == 0.1:
        only_one_correct.append(i)
print(len(only_one_correct))
print(only_one_correct)


[[1 1 1 ... 1 1 0]
 [1 1 1 ... 1 1 0]
 [1 1 1 ... 1 1 0]
 ...
 [1 1 1 ... 1 1 0]
 [1 1 1 ... 1 1 0]
 [1 1 1 ... 1 1 0]]
[1. 1. 1. ... 1. 1. 0.]
159
[90, 103, 117, 134, 288, 322, 362, 385, 391, 434, 488, 598, 604, 612, 618, 657, 675, 708, 914, 991, 996, 1097, 1292, 1347, 1435, 1454, 1457, 1553, 1558, 1618, 1627, 1676, 1733, 1755, 1815, 1817, 1859, 1868, 1882, 1923, 1928, 1971, 1983, 1995, 2176, 2183, 2270, 2283, 2520, 2610, 2644, 2728, 2773, 2787, 2790, 2796, 2809, 2816, 2830, 2914, 2965, 3008, 3065, 3074, 3233, 3302, 3384, 3417, 3495, 3506, 3575, 3615, 3628, 3856, 3867, 3907, 4014, 4015, 4070, 4100, 4121, 4159, 4191, 4201, 4217, 4291, 4295, 4302, 4325, 4327, 4450, 4473, 4494, 4524, 4583, 4668, 4674, 4694, 4720, 4789, 4811, 4924, 4931, 4966, 4986, 4993, 5154, 5239, 5273, 5333, 5340, 5436, 5440, 5502, 5563, 5589, 5612, 5614, 5670, 5703, 5778, 5794, 5909, 5920, 5936, 5993, 6002, 6131, 6167, 6201, 6252, 6282, 6314, 6339, 6340, 6367, 6374, 6429, 6460, 6566, 6716, 6737, 6758, 6760, 6775, 683

In [3]:
problems_selected = []
solutions_selected = []
positive_answers = []
negative_answers = []
for i in only_one_correct:
    problems_selected.append(problems[i])
    solutions_selected.append(solutions[i])
    # only one correct answer and one incorrect answer needed to be added
    for nth_sampling in range(len(true_n_ls)):
        if true_n_ls[nth_sampling][i] == 1:
            positive_answers.append(outputs[i][nth_sampling])
            break
    for nth_sampling in range(len(true_n_ls)):
        if true_n_ls[nth_sampling][i] == 0:
            negative_answers.append(outputs[i][nth_sampling])
            break
print(len(problems_selected))
print(len(positive_answers))
print(len(negative_answers))
# print("positive_answer:", positive_answers[-1])
# print("negative_answer:", negative_answers[-1])
# output_file_path = "/media/volume/EmbodiedAICompetition/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/gsm8k_train/10/0.1/10000/pos_neg_answers.jsonl"
# with open(output_file_path, "w", encoding="utf-8") as f:
#     for i in range(len(problems_selected)):
#         f.write(json.dumps({"problem": problems_selected[i], "solution": solutions_selected[i], "pos_answer": positive_answers[i], "neg_answer": negative_answers[i]}) + "\n")


159
159
159


In [4]:
problems_selected = []
solutions_selected = []
positive_answers = []
negative_answers = []
pos_neg_selected_path = "/media/volume/EmbodiedAICompetition/llm_steering_reasoning/results/deepseek-baseline/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/gsm8k_train/10/0.1/10000/pos_neg_answers_cleand.jsonl"
with open(pos_neg_selected_path, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        problems_selected.append(item["problem"])
        solutions_selected.append(item["solution"])
        positive_answers.append(item["pos_answer"])
        negative_answers.append(item["neg_answer"])

formatted_positive = []
formatted_negative = []
for i in range(len(problems_selected)):
    formatted_positive.append("Please reason step by step, and put your final answer within \\boxed{}.\nUser:" + problems_selected[i] + "\nAssistant: <think>" + positive_answers[i])
    formatted_negative.append("Please reason step by step, and put your final answer within \\boxed{}.\nUser:" + problems_selected[i] + "\nAssistant: <think>" + negative_answers[i])

In [5]:
# random sample 27
formatted_positive = formatted_positive[:3]
formatted_negative = formatted_negative[:3]

print(len(formatted_positive))
print(len(formatted_negative))


3
3


In [6]:
import easysteer.hidden_states as hs
from vllm import LLM
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
llm = LLM(
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    task="embed", 
    tensor_parallel_size=1,
    enforce_eager=True,
    enable_prefix_caching=False,
    enable_chunked_prefill=False
)
all_hidden_states, outputs = hs.get_all_hidden_states(llm, formatted_positive+formatted_negative)

/home/exouser/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 11-26 21:25:50 [utils.py:253] non-default args: {'task': 'embed', 'enable_prefix_caching': False, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'}
INFO 11-26 21:25:51 [model.py:657] Resolved architecture: Qwen2ForCausalLM
INFO 11-26 21:25:51 [model.py:1746] Using max model len 131072


2025-11-26 21:25:51,461	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 11-26 21:25:51 [vllm.py:414] Cudagraph is disabled under eager mode
(EngineCore_DP0 pid=615262) INFO 11-26 21:25:52 [core.py:94] Initializing a V1 LLM engine (v0.1.dev10891+ge8dee828a) with config: model='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', speculative_config=None, tokenizer='deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=131072, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.74it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.74it/s]
(EngineCore_DP0 pid=615262) 


(EngineCore_DP0 pid=615262) INFO 11-26 21:25:55 [default_loader.py:314] Loading weights took 0.67 seconds
(EngineCore_DP0 pid=615262) INFO 11-26 21:25:55 [hidden_states_model_runner_mixin.py:90] Wrapped 28 decoder layers for hidden states capture
(EngineCore_DP0 pid=615262) INFO 11-26 21:25:55 [gpu_model_runner.py:2971] Model loading took 2.9105 GiB and 1.089232 seconds
(EngineCore_DP0 pid=615262) INFO 11-26 21:25:57 [gpu_worker.py:343] Available KV cache memory: 60.12 GiB
(EngineCore_DP0 pid=615262) INFO 11-26 21:25:57 [kv_cache_utils.py:1247] GPU KV cache size: 2,251,472 tokens
(EngineCore_DP0 pid=615262) INFO 11-26 21:25:57 [kv_cache_utils.py:1252] Maximum concurrency for 131,072 tokens per request: 17.18x
(EngineCore_DP0 pid=615262) INFO 11-26 21:25:57 [core.py:238] init engine (profile, create kv cache, warmup model) took 1.85 seconds
(EngineCore_DP0 pid=615262) INFO 11-26 21:25:58 [vllm.py:414] Cudagraph is disabled under eager mode
INFO 11-26 21:25:58 [llm.py:346] Supported task

Processed prompts: 100%|██████████| 6/6 [00:00<00:00, 30.49it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


In [7]:
from easysteer.steer import extract_diffmean_control_vector, StatisticalControlVector
control_vector = extract_diffmean_control_vector(
    all_hidden_states=all_hidden_states, 
    positive_indices=list(range(len(formatted_positive))),  
    negative_indices=list(range(len(formatted_positive), len(formatted_positive)+len(formatted_negative))),  
    model_type="llama",
    token_pos=-1,
    normalize=True
)
os.makedirs("/media/volume/EmbodiedAICompetition/llm_steering_reasoning/vectors/basic/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/6", exist_ok=True)
control_vector.export_gguf("/media/volume/EmbodiedAICompetition/llm_steering_reasoning/vectors/basic/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/6/reason.gguf")

Computing DiffMean directions: 100%|██████████| 28/28 [00:00<00:00, 12275.58it/s]


In [8]:
del llm
